In [1]:
import pandas as pd
import numpy as np
import sqlite3
from scipy import stats

df = pd.read_csv("IBM-HR-Employee-Attrition.csv")

conn = sqlite3.connect(":memory:")
df.to_sql("employee_attrition", conn, index=False, if_exists="replace")

df.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [2]:
#PHASE 1: Building a sourcing filter to flag candidate profiles which are highly prone to early-stage Attrition (candidatse leaves within their
#...first 12 months)


In [3]:
# Isolate new hires with experience of 1 year or less within the company:

df_new_hires = pd.read_sql_query("""
SELECT *
FROM employee_attrition
WHERE YearsAtCompany <= 1
""", conn)

# Checking the total size of the new hire dataset:
print(f"Total new hires in dataset: {len(df_new_hires)}")


Total new hires in dataset: 215


In [4]:
# Calculating the exact attrition percentages for each unique company count

attrition_counts = pd.read_sql_query("""
SELECT
    NumCompaniesWorked,
    SUM(CASE WHEN Attrition = 'No' THEN 1 ELSE 0 END) AS No,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS Yes
FROM employee_attrition
WHERE YearsAtCompany <= 1
GROUP BY NumCompaniesWorked
ORDER BY NumCompaniesWorked
""", conn)

attrition_by_companies = attrition_counts.set_index("NumCompaniesWorked")
attrition_by_companies = attrition_by_companies.div(attrition_by_companies.sum(axis=1), axis=0) * 100

print("=== First-Year Attrition Rate by Number of Past Companies ===")
print(attrition_by_companies.round(1).astype(str) + '%')


=== First-Year Attrition Rate by Number of Past Companies ===
                        No    Yes
NumCompaniesWorked               
0                    33.3%  66.7%
1                    52.2%  47.8%
2                    70.0%  30.0%
3                    81.5%  18.5%
4                    87.5%  12.5%
5                    88.9%  11.1%
6                    64.3%  35.7%
7                    55.6%  44.4%
8                   100.0%   0.0%
9                    42.9%  57.1%


In [5]:
# Labeling experience into bins 

df_new_hires = pd.read_sql_query("""
SELECT *,
    CASE
        WHEN NumCompaniesWorked IN (0, 1) THEN 'New to Workforce / Single-Firm History'
        WHEN NumCompaniesWorked IN (2, 3, 4, 5) THEN 'Optimal Tenure Sweet Spot (2-5 Companies)'
        WHEN NumCompaniesWorked >= 6 THEN 'Chronic Job-Hopper (6+ Companies)'
        ELSE 'Unknown'
    END AS Candidate_Retention_Category
FROM employee_attrition
WHERE YearsAtCompany <= 1
""", conn)


In [6]:
retention_status = []

for status in df_new_hires['Attrition']:
    if status == 'No':
        retention_status.append('Kept Past 1 Year')
    elif status == 'Yes':
        retention_status.append('Quit Within 1 Year')
    else:
        retention_status.append('Unknown')

df_new_hires['First-Year Retention Status'] = retention_status


In [7]:
validation_counts = pd.crosstab(
    df_new_hires['Candidate_Retention_Category'],
    df_new_hires['First-Year Retention Status']
)

validation_matrix = validation_counts.div(validation_counts.sum(axis=1), axis=0) * 100

print("=== RECRUITMENT SCREENING VALIDATION: Candidate Profile vs. First-Year Turnover ===")
print(validation_matrix.round(1).astype(str) + '%')


=== RECRUITMENT SCREENING VALIDATION: Candidate Profile vs. First-Year Turnover ===
First-Year Retention Status               Kept Past 1 Year Quit Within 1 Year
Candidate_Retention_Category                                                 
Chronic Job-Hopper (6+ Companies)                    66.7%              33.3%
New to Workforce / Single-Firm History               51.0%              49.0%
Optimal Tenure Sweet Spot (2-5 Companies)            81.2%              18.8%


In [8]:
# Generating a contingency table
# Testing with Chi-Square:

contingency_table = pd.crosstab(
    df_new_hires['Candidate_Retention_Category'],
    df_new_hires['First-Year Retention Status']
)

# Run the traditional Chi-Square Test of Independence
chi2_stat, p_value, dof, expected_frequencies = stats.chi2_contingency(contingency_table)

# Print the validation results cleanly
print("=== STATISTICAL VALIDATION RESULTS ===")
print(f"Chi-Square Statistic : {chi2_stat:.4f}")
print(f"Degrees of Freedom   : {dof}")
print(f"P-Value              : {p_value:.6f}")

print("\n=== VERIFICATION DECISION ===")

if p_value < 0.05:
    print("REJECT NULL HYPOTHESIS: The candidate categories are statistically associated with early attrition.")
    print("This suggests the sourcing filter captures a meaningful pattern in the dataset.")
else:
    print("FAIL TO REJECT NULL HYPOTHESIS: The variance could be random fluctuations.")
    print("Review the category boundaries or gather a larger cohort sample.")


=== STATISTICAL VALIDATION RESULTS ===
Chi-Square Statistic : 17.5807
Degrees of Freedom   : 2
P-Value              : 0.000152

=== VERIFICATION DECISION ===
REJECT NULL HYPOTHESIS: The candidate categories are statistically associated with early attrition.
This suggests the sourcing filter captures a meaningful pattern in the dataset.


In [9]:
#PHASE 2: What is the optimal, data-driven salary baseline to offer a new hire? Is there a point where increasing the salary offer stops
#...yielding any benefits, proving that candidates are leaving due to workload/culture rather than money?


In [10]:
# Isolate new hires with experience of 1 year or less within the company:

df_new_hires = pd.read_sql_query("""
SELECT *
FROM employee_attrition
WHERE YearsAtCompany <= 1
""", conn)

print(f"Total new hires in dataset: {len(df_new_hires)}")


Total new hires in dataset: 215


In [11]:
# Finding the median income for each Job Level

job_level_medians = df_new_hires.groupby('JobLevel')['MonthlyIncome'].median().reset_index()

job_level_medians = job_level_medians.rename(
    columns={'MonthlyIncome': 'Job_Level_Median_Income'}
)

print("=== Median Monthly Income by Job Level for New Hires ===")
print(job_level_medians)


=== Median Monthly Income by Job Level for New Hires ===
   JobLevel  Job_Level_Median_Income
0         1                   2407.0
1         2                   5163.0
2         3                  10016.5
3         4                  15992.0
4         5                  18711.0


In [12]:
# Adding the job-level median income to the new hire dataset

df_new_hires = df_new_hires.merge(
    job_level_medians,
    on='JobLevel',
    how='left'
)

df_new_hires.head()


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Job_Level_Median_Income
0,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,80,0,7,3,3,0,0,0,0,2407.0
1,59,No,Travel_Rarely,1324,Research & Development,3,3,Medical,1,10,...,80,3,12,3,2,1,0,0,0,2407.0
2,30,No,Travel_Rarely,1358,Research & Development,24,1,Life Sciences,1,11,...,80,1,1,2,3,1,0,0,0,2407.0
3,22,No,Non-Travel,1123,Research & Development,16,2,Medical,1,22,...,80,2,1,2,2,1,0,0,0,2407.0
4,21,No,Travel_Rarely,391,Research & Development,15,2,Life Sciences,1,30,...,80,0,0,6,3,0,0,0,0,2407.0


In [14]:
# Labeling each new hire job levels into bins 

df_new_hires['Pay_Position'] = np.where(
    df_new_hires['MonthlyIncome'] < df_new_hires['Job_Level_Median_Income'],
    'Below Job-Level Median',
    'At/Above Job-Level Median'
)

df_new_hires[['JobLevel', 'MonthlyIncome', 'Job_Level_Median_Income', 'Pay_Position']].head()


,JobLevel,MonthlyIncome,Job_Level_Median_Income,Pay_Position
0,1,2090,2407.0,Below Job-Level Median
1,1,2670,2407.0,At/Above Job-Level Median
2,1,2693,2407.0,At/Above Job-Level Median
3,1,2935,2407.0,At/Above Job-Level Median
4,1,1232,2407.0,Below Job-Level Median


In [16]:
# Compare attrition rates by pay position

pay_position_matrix = pd.crosstab(
    df_new_hires['Pay_Position'],
    df_new_hires['Attrition'],
    normalize='index'
) * 100

print("=== New Hire Attrition by Pay Position ===")
print(pay_position_matrix.round(1).astype(str) + '%')


=== New Hire Attrition by Pay Position ===
Attrition                     No    Yes
Pay_Position                           
At/Above Job-Level Median  70.6%  29.4%
Below Job-Level Median     59.4%  40.6%


In [18]:
# Checking whether below-median pay affects higher performers differently

pay_performance_matrix = pd.crosstab(
    [df_new_hires['Pay_Position'], df_new_hires['PerformanceRating']],
    df_new_hires['Attrition'],
    normalize='index'
) * 100

print("=== New Hire Attrition by Pay Position and Performance Rating ===")
print(pay_performance_matrix.round(1).astype(str) + '%')


=== New Hire Attrition by Pay Position and Performance Rating ===
Attrition                                       No    Yes
Pay_Position              PerformanceRating              
At/Above Job-Level Median 3                  71.1%  28.9%
                          4                  68.4%  31.6%
Below Job-Level Median    3                  60.7%  39.3%
                          4                  52.9%  47.1%


In [20]:
# Statistical test: Is pay position associated with attrition?

pay_contingency = pd.crosstab(
    df_new_hires['Pay_Position'],
    df_new_hires['Attrition']
)

chi2_pay, pval_pay, dof_pay, expected_pay = stats.chi2_contingency(pay_contingency)

print("=== Pay Position Chi-Square Test ===")
print(f"Chi-Square Statistic : {chi2_pay:.4f}")
print(f"Degrees of Freedom   : {dof_pay}")
print(f"P-Value              : {pval_pay:.6f}")

if pval_pay < 0.05:
    print("DECISION: REJECT NULL HYPOTHESIS. Pay position is statistically associated with new-hire attrition.")
else:
    print("DECISION: FAIL TO REJECT. Pay position is not statistically associated with new-hire attrition in this dataset.")


=== Pay Position Chi-Square Test ===
Chi-Square Statistic : 2.4991
Degrees of Freedom   : 1
P-Value              : 0.113911
DECISION: FAIL TO REJECT. Pay position is not statistically associated with new-hire attrition in this dataset.


In [22]:
# Statistical test: Is the combination of pay position and performance rating associated with attrition?

pay_performance_contingency = pd.crosstab(
    [df_new_hires['Pay_Position'], df_new_hires['PerformanceRating']],
    df_new_hires['Attrition']
)

chi2_perf, pval_perf, dof_perf, expected_perf = stats.chi2_contingency(pay_performance_contingency)

print("=== Pay Position + Performance Chi-Square Test ===")
print(f"Chi-Square Statistic : {chi2_perf:.4f}")
print(f"Degrees of Freedom   : {dof_perf}")
print(f"P-Value              : {pval_perf:.6f}")

if pval_perf < 0.05:
    print("DECISION: REJECT NULL HYPOTHESIS. Pay position and performance level are statistically associated with new-hire attrition.")
else:
    print("DECISION: FAIL TO REJECT. The relationship between pay position, performance, and attrition is not statistically significant.")


=== Pay Position + Performance Chi-Square Test ===
Chi-Square Statistic : 3.3978
Degrees of Freedom   : 3
P-Value              : 0.334262
DECISION: FAIL TO REJECT. The relationship between pay position, performance, and attrition is not statistically significant.


In [24]:
# Isolate low-paid new hires (employees paid below median for their job level)

financially_vulnerable = df_new_hires[
    df_new_hires['Pay_Position'] == 'Below Job-Level Median'
].copy()

print(f"Financially vulnerable new hires: {len(financially_vulnerable)}")


Financially vulnerable new hires: 106


In [26]:
# Checking whether stock options reduce attrition among low-paid new hires

stock_option_matrix = pd.crosstab(
    financially_vulnerable['StockOptionLevel'],
    financially_vulnerable['Attrition'],
    normalize='index'
) * 100

print("=== Attrition of Below-Median-Paid New Hires by Stock Option Level ===")
print(stock_option_matrix.round(1).astype(str) + '%')


=== Attrition of Below-Median-Paid New Hires by Stock Option Level ===
Attrition            No    Yes
StockOptionLevel              
0                 46.3%  53.7%
1                 69.4%  30.6%
2                 80.0%  20.0%
3                 83.3%  16.7%


In [28]:
# Statistical test: Are stock options associated with attrition among low-paid new hires?

stock_option_contingency = pd.crosstab(
    financially_vulnerable['StockOptionLevel'],
    financially_vulnerable['Attrition']
)

chi2_stock, pval_stock, dof_stock, expected_stock = stats.chi2_contingency(stock_option_contingency)

print("=== Stock Option Shield Chi-Square Test ===")
print(f"Chi-Square Statistic : {chi2_stock:.4f}")
print(f"Degrees of Freedom   : {dof_stock}")
print(f"P-Value              : {pval_stock:.6f}")

if pval_stock < 0.05:
    print("DECISION: REJECT NULL HYPOTHESIS. Stock option level is statistically associated with attrition among below-median-paid new hires.")
else:
    print("DECISION: FAIL TO REJECT. Stock option level is not statistically associated with attrition among below-median-paid new hires.")


=== Stock Option Shield Chi-Square Test ===
Chi-Square Statistic : 8.5378
Degrees of Freedom   : 3
P-Value              : 0.036112
DECISION: REJECT NULL HYPOTHESIS. Stock option level is statistically associated with attrition among below-median-paid new hires.


In [30]:
# Create salaring tiers within the newly hired employees

df_new_hires['Salary_Tier'] = pd.qcut(
    df_new_hires['MonthlyIncome'],
    q=3,
    labels=['Low Salary Tier', 'Middle Salary Tier', 'High Salary Tier']
)

salary_tier_matrix = pd.crosstab(
    df_new_hires['Salary_Tier'],
    df_new_hires['Attrition'],
    normalize='index'
) * 100

print("=== New Hire Attrition by Salary Tier ===")
print(salary_tier_matrix.round(1).astype(str) + '%')


=== New Hire Attrition by Salary Tier ===
Attrition              No    Yes
Salary_Tier                     
Low Salary Tier     50.0%  50.0%
Middle Salary Tier  63.4%  36.6%
High Salary Tier    81.9%  18.1%


In [32]:
# Chi-Square test for salary tier and attrition

salary_tier_contingency = pd.crosstab(
    df_new_hires['Salary_Tier'],
    df_new_hires['Attrition']
)

chi2_salary, pval_salary, dof_salary, expected_salary = stats.chi2_contingency(salary_tier_contingency)

print("=== Salary Tier Chi-Square Test ===")
print(f"Chi-Square Statistic : {chi2_salary:.4f}")
print(f"Degrees of Freedom   : {dof_salary}")
print(f"P-Value              : {pval_salary:.6f}")

if pval_salary < 0.05:
    print("DECISION: REJECT NULL HYPOTHESIS. Salary tier is statistically associated with new-hire attrition.")
else:
    print("DECISION: FAIL TO REJECT. Salary tier is not statistically associated with new-hire attrition in this dataset.")


=== Salary Tier Chi-Square Test ===
Chi-Square Statistic : 16.3133
Degrees of Freedom   : 2
P-Value              : 0.000287
DECISION: REJECT NULL HYPOTHESIS. Salary tier is statistically associated with new-hire attrition.
